In [1]:
import pandas as pd
import glob
import os


In [2]:
path_aiming = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Aiming/filtered_data'
path_prehension = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Prehension/filtered_data'
path_visual_illusion = 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Visual Illusions/filtered_data'


In [ ]:
import os
import glob
import pandas as pd

def create_combined_dataset(base_paths, output_filename="combined_trajectory_data.csv"):
    """
    Loads, combines, and preprocesses trajectory data from multiple experimental folders,
    treating each folder as a distinct experiment.

    Args:
        base_paths (dict): A dictionary mapping a dataset name to its folder path.
        output_filename (str): The name of the CSV file to save the combined data.
    """
    all_trial_dfs = []

    print("Processing dataset folders...")
    # Loop through each provided dataset folder
    for dataset_name, base_path in base_paths.items():
        if not os.path.isdir(base_path):
            print(f"Warning: Directory not found, skipping: {base_path}")
            continue
            
        print(f"--> Processing '{dataset_name}' dataset in '{base_path}'")
        
        search_pattern = os.path.join(base_path, 's*trajData.csv')
        traj_files = glob.glob(search_pattern)
        
        if not traj_files:
            print(f"    No trajectory files found matching 's*trajData.csv' in {base_path}. Skipping.")
            continue

        for file_path in traj_files:
            print(f"    Loading {os.path.basename(file_path)}")
            try:
                df = pd.read_csv(file_path)
                # Add a source column for traceability
                df['dataset_source'] = dataset_name
                all_trial_dfs.append(df)
            except Exception as e:
                print(f"      Could not process file {file_path}: {e}")


    if not all_trial_dfs:
        print("No data was loaded. Please check your paths and file names.")
        return

    # Concatenate all individual dataframes into one large dataframe
    print("\nCombining all dataframes...")
    combined_df = pd.concat(all_trial_dfs, ignore_index=True)
    print(f"Total rows in combined data: {len(combined_df)}")

    print("Engineering the target label 'grip_strategy_label'...")

    label_config = {
        'aiming': ['visCond', 'surface', 'distance'],
        'prehension': ['visCond', 'surface', 'distance'],
        'visual_illusion': ['visCond', 'illusion', 'targetPos', 'targetSize']
    }

    # Initialize the new column
    combined_df['grip_strategy_label'] = ''

    for dataset_name, cols in label_config.items():
        mask = combined_df['dataset_source'] == dataset_name
        if mask.sum() > 0:
            conditions_str = combined_df.loc[mask, cols].fillna('NA').astype(str).agg('_'.join, axis=1)
            combined_df.loc[mask, 'grip_strategy_label'] = f"{dataset_name}_" + conditions_str
            
    print("\nCleaning the generated labels...")
    
    initial_label_count = combined_df['grip_strategy_label'].nunique()
    combined_df['grip_strategy_label'] = combined_df['grip_strategy_label'].str.replace('.csv', '', regex=False)
    
    combined_df['grip_strategy_label'] = combined_df['grip_strategy_label'].str.replace('_woord_', '_wood_', regex=False)
    
    final_label_count = combined_df['grip_strategy_label'].nunique()
    print(f"Label cleaning complete. Number of unique classes reduced from {initial_label_count} to {final_label_count}.")

    print("\n--- Sanity Check ---")
    print("Value counts for dataset sources:")
    print(combined_df['dataset_source'].value_counts())
    
    print("\nExample labels from 'aiming' dataset:")
    print(combined_df[combined_df['dataset_source'] == 'aiming']['grip_strategy_label'].value_counts().head())

    print("\nExample labels from 'prehension' dataset:")
    print(combined_df[combined_df['dataset_source'] == 'prehension']['grip_strategy_label'].value_counts().head())
    
    print("\nExample labels from 'visual_illusion' dataset:")
    print(combined_df[combined_df['dataset_source'] == 'visual_illusion']['grip_strategy_label'].value_counts().head())
    
    num_unique_labels = combined_df['grip_strategy_label'].nunique()
    print(f"\nTotal number of unique grip strategies (classes): {num_unique_labels}")

    print(f"\nSaving combined data to '{output_filename}'...")
    combined_df.to_csv(output_filename, index=False)
    print("Done! The file 'combined_trajectory_data.csv' is ready for the next step.")

In [ ]:
if __name__ == '__main__':
    paths_to_process = {
        'aiming': path_aiming,
        'prehension': path_prehension,
        'visual_illusion': path_visual_illusion
    }
    
    create_combined_dataset(paths_to_process)


Processing dataset folders...
--> Processing 'aiming' dataset in 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Aiming/filtered_data'
    Loading s001trajData.csv
    Loading s002trajData.csv
    Loading s003trajData.csv
    Loading s004trajData.csv
    Loading s005trajData.csv
    Loading s006trajData.csv
    Loading s007trajData.csv
    Loading s008trajData.csv
    Loading s009trajData.csv
    Loading s010trajData.csv
    Loading s011trajData.csv
    Loading s012trajData.csv
    Loading s013trajData.csv
    Loading s014trajData.csv
    Loading s015trajData.csv
    Loading s016trajData.csv
    Loading s017trajData.csv
    Loading s018trajData.csv
--> Processing 'prehension' dataset in 'C:/CourseWork/Dissertation Classifying grip strategies using machine learning/data/01_raw/Prehension/filtered_data'
    Loading s001trajData.csv
    Loading s002trajData.csv
    Loading s003trajData.csv
    Loading s004trajData.csv
    Loading s005trajData.csv